In [1]:
import os
import random
import time
import numpy as np
import threadpoolctl

random.seed(0)
np.random.seed(0)
DTYPE = np.float32


In [2]:
def xavier(shape, dtype=DTYPE):
    fan_out = shape[0]
    fan_in = shape[1] if len(shape) > 1 else shape[0]
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, size=shape).astype(dtype)


def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)


In [ ]:
PARAM_NAMES = [
    "Emb",
    "Wxh_enc", "Whh_enc", "bh_enc",
    "W_s", "b_s", "W_h", "b_h", "v", "bv",
    "Wxh_dec", "Whh_dec", "bh_dec",
    "W_out", "b_out",
]


class BahdanauSeq2Seq:
    def __init__(self, vocab_size, emb_dim, hidden_size, dtype=DTYPE):
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_size = hidden_size
        self.dtype = dtype

        self.Emb = xavier((vocab_size, emb_dim), dtype)

        self.Wxh_enc = xavier((hidden_size, emb_dim), dtype)
        self.Whh_enc = xavier((hidden_size, hidden_size), dtype)
        self.bh_enc = np.zeros(hidden_size, dtype=dtype)

        self.W_s = xavier((hidden_size, hidden_size), dtype)
        self.b_s = np.zeros(hidden_size, dtype=dtype)
        self.W_h = xavier((hidden_size, hidden_size), dtype)
        self.b_h = np.zeros(hidden_size, dtype=dtype)
        self.v = xavier((hidden_size,), dtype)
        self.bv = np.zeros(1, dtype=dtype)

        self.Wxh_dec = xavier((hidden_size, emb_dim + hidden_size), dtype)
        self.Whh_dec = xavier((hidden_size, hidden_size), dtype)
        self.bh_dec = np.zeros(hidden_size, dtype=dtype)

        self.W_out = xavier((vocab_size, hidden_size * 2 + emb_dim), dtype)
        self.b_out = np.zeros(vocab_size, dtype=dtype)

        for name in PARAM_NAMES:
            setattr(self, "d" + name, np.zeros_like(getattr(self, name)))


    def forward(self, src_ids, tgt_ids, teacher_forcing_ratio=1.0):
        H_sz = self.hidden_size
        T_src = len(src_ids)

        H = np.zeros((T_src, H_sz), dtype=self.dtype)
        enc_cache = []
        h_prev = np.zeros(H_sz, dtype=self.dtype)
        for t in range(T_src):
            x_t = self.Emb[src_ids[t]]
            z = self.Wxh_enc @ x_t + self.Whh_enc @ h_prev + self.bh_enc
            h_t = np.tanh(z)
            H[t] = h_t
            enc_cache.append((src_ids[t], x_t, h_prev, h_t))
            h_prev = h_t

        hidden = H[-1].copy()
        dec_cache = []
        loss = 0.0
        input_token = int(tgt_ids[0])
        for t in range(len(tgt_ids) - 1):
            target_id = int(tgt_ids[t + 1])
            embedded = self.Emb[input_token]

            query_proj = self.W_s @ hidden + self.b_s              
            keys_proj = H @ self.W_h.T + self.b_h
            U = np.tanh(keys_proj + query_proj)  
            scores = U @ self.v + self.bv                          
            alpha = softmax(scores)                                
            context = alpha @ H                                    

            rnn_input = np.concatenate([embedded, context])
            z_dec = self.Wxh_dec @ rnn_input + self.Whh_dec @ hidden + self.bh_dec
            hidden_new = np.tanh(z_dec)

            pred_input = np.concatenate([hidden_new, context, embedded])
            logits = self.W_out @ pred_input + self.b_out
            probs = softmax(logits)
            loss += -np.log(probs[target_id] + 1e-9)

            dec_cache.append(dict(
                input_token=input_token, embedded=embedded, hidden_prev=hidden,
                U=U, alpha=alpha, context=context,
                rnn_input=rnn_input, hidden_new=hidden_new,
                pred_input=pred_input, probs=probs, target_id=target_id,
            ))

            use_tf = random.random() < teacher_forcing_ratio
            input_token = target_id if use_tf else int(np.argmax(logits))
            hidden = hidden_new

        loss /= max(1, len(tgt_ids) - 1)
        return loss, H, enc_cache, dec_cache


    def backward(self, H, enc_cache, dec_cache):
        T_src, H_sz = H.shape
        E = self.emb_dim
        n = len(dec_cache)

        dH = np.zeros_like(H)
        dhidden_next = np.zeros(H_sz, dtype=self.dtype)

        for t in reversed(range(n)):
            c = dec_cache[t]

            # undoes: probs = softmax(logits); loss += -log(probs[target])
            # closed form for softmax+cross-entropy together: predicted - one_hot(target)
            dlogits = c["probs"].copy()
            dlogits[c["target_id"]] -= 1.0
            dlogits /= n

            # undoes: logits = W_out @ pred_input + b_out
            self.dW_out += np.outer(dlogits, c["pred_input"])
            self.db_out += dlogits
            dpred_input = self.W_out.T @ dlogits

            # undoes: pred_input = concatenate([hidden_new, context, embedded])
            # (slicing undoes concatenation - each chunk gets its own slice of the gradient)
            dhidden_new = dpred_input[:H_sz] + dhidden_next   # hidden_new is ALSO next step's input -> sum
            dcontext = dpred_input[H_sz:2 * H_sz]
            dembedded = dpred_input[2 * H_sz:2 * H_sz + E].copy()

            # undoes: hidden_new = tanh(Wxh_dec @ rnn_input + Whh_dec @ hidden_prev + bh_dec)
            dz_dec = dhidden_new * (1 - c["hidden_new"] ** 2)          # tanh'(z) = 1 - tanh(z)^2
            self.dWxh_dec += np.outer(dz_dec, c["rnn_input"])
            self.dWhh_dec += np.outer(dz_dec, c["hidden_prev"])
            self.dbh_dec += dz_dec
            drnn_input = self.Wxh_dec.T @ dz_dec
            dhidden_prev = self.Whh_dec.T @ dz_dec                     # gradient into "hidden" BEFORE this step

            # undoes: rnn_input = concatenate([embedded, context])
            dembedded += drnn_input[:E]
            dcontext = dcontext + drnn_input[E:]

            # undoes attention: context = alpha @ H  (context path)
            alpha, U = c["alpha"], c["U"]
            dH_from_context = np.outer(alpha, dcontext)
            dalpha = H @ dcontext
            # undoes: alpha = softmax(scores)
            dscores = alpha * (dalpha - np.sum(alpha * dalpha))

            # undoes: scores = U @ v + bv
            self.dv += dscores @ U
            self.dbv += np.sum(dscores)
            dU = np.outer(dscores, self.v)

            # undoes: U = tanh(keys_proj + query_proj)
            dz_attn = dU * (1 - U ** 2)
            dz_attn_sum = dz_attn.sum(axis=0)     # query_proj was broadcast over all T_src rows -> sum back

            # undoes: query_proj = W_s @ hidden + b_s   (hidden = c["hidden_prev"] here)
            self.dW_s += np.outer(dz_attn_sum, c["hidden_prev"])
            self.db_s += dz_attn_sum
            ddecoder_hidden = self.W_s.T @ dz_attn_sum

            # undoes: keys_proj = H @ W_h.T + b_h   (attention-key path, H used a SECOND time)
            self.dW_h += dz_attn.T @ H
            self.db_h += dz_attn_sum
            dH_from_keys = dz_attn @ self.W_h

            # H was used twice in attention (context path + key path) -> gradients ADD
            dH += dH_from_context + dH_from_keys
            # "hidden" (the query) was used twice in this decoder step (RNN's h_prev AND
            # attention's query) -> gradients ADD
            dhidden_prev = dhidden_prev + ddecoder_hidden

            # undoes: embedded = self.Emb[input_token]  (lookup -> scatter-add into that row)
            self.dEmb[c["input_token"]] += dembedded

            dhidden_next = dhidden_prev   # becomes "next-step" gradient for the earlier timestep

        # =============== ENCODER, latest timestep to earliest ===============
        # dhidden_next here is the gradient that reached H[-1] for being the decoder's
        # initial hidden state (the "bridge") - on top of dH[t], already collected above,
        # from every encoder position being attended over at every decoder step.
        dh_next = dhidden_next
        for t in reversed(range(T_src)):
            x_id, x_t, h_prev, h_t = enc_cache[t]
            dh_total = dH[t] + dh_next          # H[t] used by attention (dH[t]) AND by H[t+1] (dh_next) -> sum

            # undoes: h_t = tanh(Wxh_enc @ x_t + Whh_enc @ h_prev + bh_enc)
            dz_enc = dh_total * (1 - h_t ** 2)
            self.dWxh_enc += np.outer(dz_enc, x_t)
            self.dWhh_enc += np.outer(dz_enc, h_prev)
            self.dbh_enc += dz_enc
            dx_t = self.Wxh_enc.T @ dz_enc
            dh_next = self.Whh_enc.T @ dz_enc    # carries back to h_{t-1}

            # undoes: x_t = self.Emb[src_ids[t]]
            self.dEmb[x_id] += dx_t


    def greedy_decode(self, src_ids, sos_id, eos_id, max_len):
        H_sz = self.hidden_size
        h_prev = np.zeros(H_sz, dtype=self.dtype)
        H = np.zeros((len(src_ids), H_sz), dtype=self.dtype)
        for t in range(len(src_ids)):
            x_t = self.Emb[src_ids[t]]
            h_prev = np.tanh(self.Wxh_enc @ x_t + self.Whh_enc @ h_prev + self.bh_enc)
            H[t] = h_prev

        hidden = H[-1]
        token = sos_id
        tokens = []
        for _ in range(max_len):
            embedded = self.Emb[token]
            query_proj = self.W_s @ hidden + self.b_s
            keys_proj = H @ self.W_h.T + self.b_h
            U = np.tanh(keys_proj + query_proj)
            alpha = softmax(U @ self.v + self.bv)
            context = alpha @ H

            rnn_input = np.concatenate([embedded, context])
            hidden = np.tanh(self.Wxh_dec @ rnn_input + self.Whh_dec @ hidden + self.bh_dec)

            pred_input = np.concatenate([hidden, context, embedded])
            logits = self.W_out @ pred_input + self.b_out
            token = int(np.argmax(logits))
            if token == eos_id:
                break
            tokens.append(token)
        return tokens

    def zero_grad(self):
        for name in PARAM_NAMES:
            getattr(self, "d" + name).fill(0)

    def parameters(self):
        return [(getattr(self, n), getattr(self, "d" + n)) for n in PARAM_NAMES]


## Gradient check

Before trusting `backward` on real data: perturb one parameter entry at a time by
`epsilon`, re-run `forward` to get the numeric slope `(loss(+eps) - loss(-eps)) /
(2*eps)`, and compare against the analytic gradient `backward` computed. Uses a tiny
float64 model (float32 rounding noise would swamp a finite-difference check at this
scale).

In [4]:
def gradient_check():
    rng_state = random.getstate()
    np_state = np.random.get_state()
    try:
        random.seed(1)
        np.random.seed(1)
        vocab, emb, hid = 20, 5, 7
        m = BahdanauSeq2Seq(vocab, emb, hid, dtype=np.float64)
        src_ids = np.random.randint(4, vocab, size=6)
        tgt_ids = np.concatenate([[1], np.random.randint(4, vocab, size=4), [2]])

        loss, H, enc_cache, dec_cache = m.forward(src_ids, tgt_ids, teacher_forcing_ratio=1.0)
        m.zero_grad()
        m.backward(H, enc_cache, dec_cache)

        eps = 1e-5
        worst_rel_err = 0.0
        checked = 0
        for name in PARAM_NAMES:
            p = getattr(m, name)
            g = getattr(m, "d" + name)
            flat_p, flat_g = p.reshape(-1), g.reshape(-1)
            idxs = np.random.choice(len(flat_p), size=min(3, len(flat_p)), replace=False)
            for idx in idxs:
                orig = flat_p[idx]
                flat_p[idx] = orig + eps
                loss_plus, *_ = m.forward(src_ids, tgt_ids, teacher_forcing_ratio=1.0)
                flat_p[idx] = orig - eps
                loss_minus, *_ = m.forward(src_ids, tgt_ids, teacher_forcing_ratio=1.0)
                flat_p[idx] = orig

                numeric = (loss_plus - loss_minus) / (2 * eps)
                analytic = flat_g[idx]
                rel_err = abs(numeric - analytic) / max(1e-8, abs(numeric) + abs(analytic))
                worst_rel_err = max(worst_rel_err, rel_err)
                checked += 1
        print(f"Checked {checked} parameter entries across {len(PARAM_NAMES)} arrays.")
        print(f"Worst relative error: {worst_rel_err:.2e}")
        assert worst_rel_err < 1e-3, "gradient check failed"
        print("Gradient check PASSED - backward() matches finite differences.")
    finally:
        random.setstate(rng_state)
        np.random.set_state(np_state)


gradient_check()


Checked 43 parameter entries across 15 arrays.
Worst relative error: 6.47e-08
Gradient check PASSED - backward() matches finite differences.


## Training demo: same overfit config as `bahdanau_numpy_rnn.ipynb` section 8.5

Loads the real BBC News Summary data, takes the same **100-pair random subset**
(`random.seed(0)`, `random.sample(pairs, 100)`) and builds a vocab from just that
subset - identical setup to the class-based notebook's overfit sanity check, so the
two are directly comparable. Same dims (`EMBEDDING_SIZE=64`, `HIDDEN_SIZE=128`), same
optimizer settings (`LEARNING_RATE=0.001`, `WEIGHT_DECAY=1e-5`, `GRAD_CLIP=5.0`), same
100 fixed epochs with no val/early stopping, same single-threaded-BLAS trick to keep
it fast.

In [5]:
base_dir = "../BBC News Summary"
articles_dir = os.path.join(base_dir, "News Articles")
summaries_dir = os.path.join(base_dir, "Summaries")

pairs = []
max_article_len = 40
max_summary_len = 10

categories = os.listdir(articles_dir)
for cat in categories:
    cat_article_dir = os.path.join(articles_dir, cat)
    cat_summary_dir = os.path.join(summaries_dir, cat)
    if not os.path.isdir(cat_article_dir):
        continue
    for fname in sorted(os.listdir(cat_article_dir)):
        if not fname.endswith(".txt"):
            continue
        article_path = os.path.join(cat_article_dir, fname)
        summary_path = os.path.join(cat_summary_dir, fname)
        if not os.path.exists(summary_path):
            continue
        with open(article_path, "r", encoding="latin-1") as f:
            lines = f.read().strip().split("\n")
        article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
        article_words = article.split()[:max_article_len]
        if len(article_words) < 5:
            continue
        with open(summary_path, "r", encoding="latin-1") as f:
            summary = f.read().strip()
        summary_words = summary.split()[:max_summary_len]
        if len(summary_words) < 3:
            continue
        pairs.append((" ".join(article_words).lower(), " ".join(summary_words).lower()))

print(f"Loaded {len(pairs)} article-summary pairs")

# --- same 100-pair overfit subset as bahdanau_numpy_rnn.ipynb section 8.5 ---
OVERFIT_N = 100
random.seed(0)
overfit_pairs = random.sample(pairs, OVERFIT_N)

SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
words = set()
for a, b in overfit_pairs:
    words.update(a.split())
    words.update(b.split())
vocab = SPECIAL + sorted(words)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)


def encode(text, sos=False, eos=False):
    ids = ([stoi["<SOS>"]] if sos else []) + [stoi.get(w, stoi["<UNK>"]) for w in text.split()] + ([stoi["<EOS>"]] if eos else [])
    return np.array(ids, dtype=np.int64)


data = [(encode(a), encode(b, sos=True, eos=True)) for a, b in overfit_pairs]
TARGET_LEN = max_summary_len + 2
print(f"Overfit subset: {len(data)} pairs, vocab size: {vocab_size}")


Loaded 2225 article-summary pairs
Overfit subset: 100 pairs, vocab size: 2082


In [6]:
EMBEDDING_SIZE = 64     # matches OVERFIT_EMBEDDING_SIZE in bahdanau_numpy_rnn.ipynb
HIDDEN_SIZE = 128       # matches OVERFIT_HIDDEN_SIZE
EPOCHS = 100            # fixed count, no early stopping - we WANT to overfit here
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 5.0

model = BahdanauSeq2Seq(vocab_size, EMBEDDING_SIZE, HIDDEN_SIZE, dtype=DTYPE)


class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.weight_decay = weight_decay
        self.m = [np.zeros_like(p) for p, g in parameters]
        self.v = [np.zeros_like(p) for p, g in parameters]
        self.t = 0

    def step(self, parameters):
        self.t += 1
        for i, (p, g) in enumerate(parameters):
            g = g + self.weight_decay * p
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p -= (self.lr * m_hat / (np.sqrt(v_hat) + self.eps)).astype(p.dtype)


def clip_grads(parameters, max_norm=GRAD_CLIP):
    for p, g in parameters:
        norm = np.linalg.norm(g)
        if norm > max_norm:
            g *= max_norm / (norm + 1e-9)


optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

start = time.time()
with threadpoolctl.threadpool_limits(limits=1, user_api="blas"):
    # single-threaded BLAS - these matmuls are tiny, thread spawn/sync overhead would
    # otherwise dominate (same fix as bahdanau_numpy_rnn.ipynb section 8.5)
    for epoch in range(1, EPOCHS + 1):
        random.shuffle(data)
        total_loss = 0.0
        for src_ids, tgt_ids in data:
            # no teacher-forcing decay - fully teacher-forced throughout, we WANT
            # this to overfit as easily as possible (matches overfit_train)
            loss, H, enc_cache, dec_cache = model.forward(src_ids, tgt_ids, teacher_forcing_ratio=1.0)
            model.zero_grad()
            model.backward(H, enc_cache, dec_cache)
            params = model.parameters()
            clip_grads(params)
            optimizer.step(params)
            total_loss += loss
        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            elapsed = time.time() - start
            print(f"Epoch {epoch}/{EPOCHS}  train loss: {total_loss / len(data):.4f}  elapsed: {elapsed:.1f}s")

print(f"\nDone in {time.time() - start:.1f}s")


Epoch 1/100  train loss: 6.9352  elapsed: 2.0s


Epoch 10/100  train loss: 2.9905  elapsed: 16.9s


Epoch 20/100  train loss: 0.4250  elapsed: 38.9s


Epoch 30/100  train loss: 0.0518  elapsed: 61.4s


Epoch 40/100  train loss: 0.0185  elapsed: 83.2s


...

Epoch 60/100  train loss: 0.2125  elapsed: 126.8s


Epoch 70/100  train loss: 0.0141  elapsed: 148.1s


Epoch 80/100  train loss: 0.0065  elapsed: 168.9s


Epoch 90/100  train loss: 0.0042  elapsed: 190.5s


Epoch 100/100  train loss: 0.0031  elapsed: 211.5s

Done in 211.5s


In [7]:
def summarize_ids(src_ids):
    tokens = model.greedy_decode(src_ids, stoi["<SOS>"], stoi["<EOS>"], max_len=TARGET_LEN)
    return " ".join(itos[t] for t in tokens if t not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"]))


def ids_to_text(ids):
    return " ".join(itos[int(i)] for i in ids if int(i) not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"]))


exact = 0
sample = data[:15]
for src_ids, tgt_ids in sample:
    predicted = summarize_ids(src_ids)
    actual = ids_to_text(tgt_ids)
    if predicted == actual:
        exact += 1
    print(f"Article:   {ids_to_text(src_ids)}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}")
    print("=" * 60)

print(f"Exact matches: {exact}/{len(sample)}")


Article:   fiat is to stop making six-cylinder petrol engines for its sporty alfa romeo subsidiary, unions at the italian carmaker have said. the unions claim fiat is to close the fiat powertrain plant at arese near milan and instead source six-cylinder
Actual:    fiat is to stop making six-cylinder petrol engines for its
Predicted: fiat is to stop making six-cylinder petrol engines for its
Article:   the stage adaptation of children's film mary poppins has had its opening night in london's west end. sir cameron mackintosh's lavish production, which has cost â£9m to bring to the stage, was given a 10-minute standing ovation. lead actress laura
Actual:    mary poppins was originally created by author pamela travers, who
Predicted: mary poppins was originally created by author pamela travers, who
Article:   the ultimate prize of 10 downing street may continue to elude him but, as he prepares to deliver a record-breaking ninth budget, gordon brown can at least console himself with the tho

Article:   organisers say this year's berlin film festival, which opens on thursday with period epic man to man, will celebrate a revitalised european cinema. of the 21 films in competition for the golden and silver bear awards, more than half are
Actual:    festival director dieter kosslick says this strong showing signals "a
Predicted: festival director dieter kosslick says this strong showing signals "a
Exact matches: 15/15
